# MultiGuard: Full Training Pipeline on Kaggle T4 GPU

**Self-contained notebook** for training the MultiGuard multimodal hateful meme detection system.

## Pipeline Stages
1. **Baseline** — CLIP ViT-B/16 + RoBERTa + Late Fusion (10 epochs)
2. **Fusion Ablation** — Cross-Attention & Tensor Fusion comparison (15 epochs each)
3. **Knowledge Distillation** — Teacher→Student compression (20 epochs)

## Prerequisites
1. **Enable GPU**: Settings → Accelerator → GPU T4 x2 (or T4)
2. **Add Dataset**: Click "+ Add Data" → search "hateful memes" → add the dataset
   - Expected path: `/kaggle/input/hateful-memes/`
3. **Internet**: Enable internet access for downloading pretrained models

## Hardware Requirements
- **GPU**: NVIDIA T4 (16GB VRAM) — Kaggle free tier
- **RAM**: 13GB system RAM
- **Precision**: FP16 (T4 does not support BF16)

In [ ]:
# Cell 2: Install Dependencies
!pip install -q transformers datasets pillow loguru scikit-learn albumentations omegaconf

In [ ]:
# Cell 3: GPU Check & Imports
import os
import json
import random
import time
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor
from loguru import logger

warnings.filterwarnings("ignore")
%matplotlib inline

# GPU check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_total:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("WARNING: No GPU detected! Enable GPU in Kaggle settings.")

print(f"PyTorch: {torch.__version__}")

In [ ]:
# Cell 4: Configuration
CONFIG = {
    # Paths
    "data_dir": "/kaggle/input/hateful-memes",
    "output_dir": "/kaggle/working",
    "seed": 42,

    # Stage 1: Baseline
    "baseline": {
        "vision_model": "openai/clip-vit-base-patch16",
        "text_model": "roberta-base",
        "fusion": "late_fusion",
        "vision_dim": 512,  # CLIP ViT-B/16 projection dim
        "text_dim": 768,
        "hidden_dims": [512, 256],
        "num_labels": 2,
        "epochs": 10,
        "batch_size": 16,
        "grad_accum_steps": 4,
        "lr": 1e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "max_seq_length": 77,  # CLIP tokenizer max
    },

    # Stage 2: Fusion Ablation
    "fusion_ablation": {
        "epochs": 15,
        "batch_size": 8,
        "grad_accum_steps": 8,
        "lr": 5e-5,
        "fusions": ["cross_attention", "tensor_fusion"],
        # Cross-attention params
        "cross_attention": {
            "hidden_dim": 512,
            "num_heads": 8,
            "num_layers": 2,  # Reduced for T4
            "feedforward_dim": 1024,
        },
        # Tensor fusion params
        "tensor_fusion": {
            "vision_dim": 512,
            "text_dim": 768,
            "hidden_dim": 512,
            "rank": 32,
        },
    },

    # Stage 3: Knowledge Distillation
    "distillation": {
        "student_vision_model": "google/efficientnet-b0",
        "student_text_model": "distilroberta-base",
        "student_vision_dim": 1280,  # EfficientNet-B0 output
        "student_text_dim": 768,
        "epochs": 20,
        "batch_size": 32,
        "grad_accum_steps": 2,
        "lr": 1e-3,
        "temperature": 4.0,
        "alpha": 0.5,
    },
}

print("Configuration loaded.")
print(f"Data dir: {CONFIG['data_dir']}")
print(f"Output dir: {CONFIG['output_dir']}")

In [ ]:
# Cell 5: Utilities — set_seed, get_device, log_vram_usage

def set_seed(seed: int = 42) -> None:
    """Set random seed for reproducibility across torch, numpy, random, and CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    """Get the best available device (CUDA > CPU)."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        logger.info("Using CPU")
    return device


def log_vram_usage(tag: str = "") -> None:
    """Log current VRAM usage."""
    if not torch.cuda.is_available():
        return
    allocated = torch.cuda.memory_allocated(0) / (1024**3)
    reserved = torch.cuda.memory_reserved(0) / (1024**3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    logger.info(
        f"[VRAM {tag}] Allocated: {allocated:.2f}GB | "
        f"Reserved: {reserved:.2f}GB | Total: {total:.2f}GB"
    )


# Initialize
set_seed(CONFIG["seed"])
DEVICE = get_device()
log_vram_usage("init")
print(f"Device: {DEVICE}")

In [ ]:
# Cell 6: Dataset Path Setup & Load JSONL Annotations

DATA_DIR = Path(CONFIG["data_dir"])

# Hateful Memes dataset structure varies by Kaggle upload.
# Common structures:
#   /kaggle/input/hateful-memes/data/  (with img/ subfolder)
#   /kaggle/input/hateful-memes/       (flat)
# We auto-detect the layout.

def find_data_paths(base_dir: Path) -> dict:
    """Auto-detect dataset layout and return paths."""
    candidates = [base_dir, base_dir / "data", base_dir / "hateful_memes"]
    
    for d in candidates:
        train_jsonl = None
        # Look for train.jsonl or similar
        for name in ["train.jsonl", "train.json"]:
            if (d / name).exists():
                train_jsonl = d / name
                break
        if train_jsonl:
            # Find image directory
            img_dir = d / "img" if (d / "img").exists() else d
            return {
                "root": d,
                "img_dir": img_dir,
                "train": train_jsonl,
                "dev": d / "dev_seen.jsonl" if (d / "dev_seen.jsonl").exists() else d / "dev.jsonl",
                "test": d / "test_seen.jsonl" if (d / "test_seen.jsonl").exists() else d / "test.jsonl",
            }
    
    # Fallback: list what's available
    print("Available files in data directory:")
    for f in sorted(base_dir.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(base_dir)}")
    raise FileNotFoundError(f"Could not find dataset files in {base_dir}")


def load_jsonl(path: Path) -> list[dict]:
    """Load JSONL file into list of dicts."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


paths = find_data_paths(DATA_DIR)
print(f"Data root: {paths['root']}")
print(f"Image dir: {paths['img_dir']}")

train_data = load_jsonl(paths["train"])
print(f"Train samples: {len(train_data)}")

if paths["dev"].exists():
    val_data = load_jsonl(paths["dev"])
    print(f"Val samples: {len(val_data)}")
else:
    # Split train into train/val
    split_idx = int(0.9 * len(train_data))
    val_data = train_data[split_idx:]
    train_data = train_data[:split_idx]
    print(f"Split: Train={len(train_data)}, Val={len(val_data)}")

if paths["test"].exists():
    test_data = load_jsonl(paths["test"])
    print(f"Test samples: {len(test_data)}")
else:
    test_data = val_data  # Use val as test fallback
    print("No separate test set found, using val as test.")

# Show sample
print(f"\nSample entry: {train_data[0]}")

In [ ]:
# Cell 7: Image Transforms

# CLIP normalization stats
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

print("Image transforms defined.")

In [ ]:
# Cell 8: HatefulMemesDataset

class HatefulMemesDataset(Dataset):
    """Dataset for Hateful Memes challenge."""

    def __init__(
        self,
        data: list[dict],
        img_dir: Path,
        tokenizer: AutoTokenizer,
        transform: transforms.Compose,
        max_length: int = 77,
    ):
        self.data = data
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        entry = self.data[idx]

        # Load image
        img_path = self.img_dir / entry["img"]
        if not img_path.exists():
            # Try without subdirectory prefix
            img_path = self.img_dir / Path(entry["img"]).name
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.transform(image)

        # Tokenize text
        text = entry.get("text", "")
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Label
        label = entry.get("label", 0)

        return {
            "pixel_values": pixel_values,
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


print("HatefulMemesDataset class defined.")

In [ ]:
# Cell 9: Create DataLoaders

# Load tokenizer (shared across stages, RoBERTa-compatible)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

train_dataset = HatefulMemesDataset(
    data=train_data,
    img_dir=paths["img_dir"],
    tokenizer=tokenizer,
    transform=train_transform,
    max_length=CONFIG["baseline"]["max_seq_length"],
)

val_dataset = HatefulMemesDataset(
    data=val_data,
    img_dir=paths["img_dir"],
    tokenizer=tokenizer,
    transform=val_transform,
    max_length=CONFIG["baseline"]["max_seq_length"],
)

test_dataset = HatefulMemesDataset(
    data=test_data,
    img_dir=paths["img_dir"],
    tokenizer=tokenizer,
    transform=val_transform,
    max_length=CONFIG["baseline"]["max_seq_length"],
)


def create_loaders(batch_size: int, num_workers: int = 2) -> tuple:
    """Create train/val/test DataLoaders with given batch size."""
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, val_loader, test_loader


# Verify with a single batch
tmp_loader, _, _ = create_loaders(batch_size=4)
sample_batch = next(iter(tmp_loader))
print(f"pixel_values: {sample_batch['pixel_values'].shape}")
print(f"input_ids: {sample_batch['input_ids'].shape}")
print(f"attention_mask: {sample_batch['attention_mask'].shape}")
print(f"labels: {sample_batch['labels'].shape}")
del tmp_loader, sample_batch

In [ ]:
# Cell 10: Vision Encoder

class VisionEncoder(nn.Module):
    """Vision encoder wrapping CLIP vision model.
    Extracts image features via CLIP's get_image_features().
    """

    def __init__(self, model_id: str = "openai/clip-vit-base-patch16"):
        super().__init__()
        self.model_id = model_id
        self.clip_model = CLIPModel.from_pretrained(model_id)
        # Only keep the vision part
        self.vision_model = self.clip_model.vision_model
        self.visual_projection = self.clip_model.visual_projection
        # Clean up text components to save memory
        del self.clip_model.text_model
        del self.clip_model.text_projection
        self.output_dim = self.visual_projection.out_features
        logger.info(f"VisionEncoder: {model_id}, output_dim={self.output_dim}")

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Extract image features [B, output_dim]."""
        vision_outputs = self.vision_model(pixel_values=pixel_values)
        pooled = vision_outputs.pooler_output  # [B, hidden_size]
        projected = self.visual_projection(pooled)  # [B, projection_dim]
        return projected


print("VisionEncoder defined.")

In [ ]:
# Cell 11: Text Encoder

class TextEncoder(nn.Module):
    """Text encoder wrapping RoBERTa/DistilRoBERTa.
    Extracts CLS token features.
    """

    def __init__(self, model_id: str = "roberta-base", hidden_size: int = 768):
        super().__init__()
        self.model_id = model_id
        self.encoder = AutoModel.from_pretrained(model_id)
        self.output_dim = hidden_size
        logger.info(f"TextEncoder: {model_id}, output_dim={self.output_dim}")

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """Extract text features [B, hidden_size]."""
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0]  # CLS token


print("TextEncoder defined.")

In [ ]:
# Cell 12: Fusion Modules — Late, Cross-Attention, Tensor

class LateFusion(nn.Module):
    """Late fusion via concatenation and MLP."""

    def __init__(
        self,
        vision_dim: int = 512,
        text_dim: int = 768,
        hidden_dims: list[int] | None = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        hidden_dims = hidden_dims or [512, 256]
        input_dim = vision_dim + text_dim

        layers: list[nn.Module] = []
        for dim in hidden_dims:
            layers.extend([
                nn.Linear(input_dim, dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            input_dim = dim

        self.mlp = nn.Sequential(*layers)
        self.output_dim = hidden_dims[-1]

    def forward(
        self, vision_features: torch.Tensor, text_features: torch.Tensor
    ) -> torch.Tensor:
        combined = torch.cat([vision_features, text_features], dim=-1)
        return self.mlp(combined)


class CrossAttentionFusion(nn.Module):
    """Bidirectional cross-attention fusion with gated combination."""

    def __init__(
        self,
        hidden_dim: int = 512,
        num_heads: int = 8,
        num_layers: int = 2,
        feedforward_dim: int = 1024,
        dropout: float = 0.1,
        # Input projection dims (maps different-sized inputs to hidden_dim)
        vision_dim: int = 512,
        text_dim: int = 768,
    ):
        super().__init__()
        self.output_dim = hidden_dim

        # Project inputs to common hidden_dim
        self.vision_proj = nn.Linear(vision_dim, hidden_dim) if vision_dim != hidden_dim else nn.Identity()
        self.text_proj = nn.Linear(text_dim, hidden_dim) if text_dim != hidden_dim else nn.Identity()

        # Vision-to-text cross-attention
        self.v2t_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=hidden_dim, nhead=num_heads,
                dim_feedforward=feedforward_dim, dropout=dropout,
                batch_first=True,
            )
            for _ in range(num_layers)
        ])

        # Text-to-vision cross-attention
        self.t2v_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=hidden_dim, nhead=num_heads,
                dim_feedforward=feedforward_dim, dropout=dropout,
                batch_first=True,
            )
            for _ in range(num_layers)
        ])

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid(),
        )

    def forward(
        self, vision_features: torch.Tensor, text_features: torch.Tensor
    ) -> torch.Tensor:
        v = self.vision_proj(vision_features).unsqueeze(1)
        t = self.text_proj(text_features).unsqueeze(1)

        v2t = v
        for layer in self.v2t_layers:
            v2t = layer(v2t, t)

        t2v = t
        for layer in self.t2v_layers:
            t2v = layer(t2v, v)

        v2t = v2t.squeeze(1)
        t2v = t2v.squeeze(1)

        gate_input = torch.cat([v2t, t2v], dim=-1)
        gate_weight = self.gate(gate_input)
        fused = gate_weight * v2t + (1 - gate_weight) * t2v
        return fused


class TensorFusion(nn.Module):
    """Low-rank tensor fusion network."""

    def __init__(
        self,
        vision_dim: int = 512,
        text_dim: int = 768,
        hidden_dim: int = 512,
        rank: int = 32,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.output_dim = hidden_dim
        self.vision_factor = nn.Linear(vision_dim, rank, bias=False)
        self.text_factor = nn.Linear(text_dim, rank, bias=False)
        self.fusion_weights = nn.Linear(rank, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(
        self, vision_features: torch.Tensor, text_features: torch.Tensor
    ) -> torch.Tensor:
        v_factor = self.vision_factor(vision_features)
        t_factor = self.text_factor(text_features)
        fused = v_factor * t_factor
        fused = self.fusion_weights(fused)
        fused = self.dropout(fused)
        fused = self.layer_norm(fused)
        return fused


print("Fusion modules defined: LateFusion, CrossAttentionFusion, TensorFusion")

In [ ]:
# Cell 13: Classification Head & Focal Loss

class ClassificationHead(nn.Module):
    """Classification head for multimodal content analysis."""

    def __init__(
        self,
        input_dim: int = 256,
        num_labels: int = 2,
        hidden_dims: list[int] | None = None,
        dropout: float = 0.3,
    ):
        super().__init__()
        hidden_dims = hidden_dims or [256]

        layers: list[nn.Module] = []
        in_dim = input_dim
        for dim in hidden_dims:
            layers.extend([
                nn.Linear(in_dim, dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            in_dim = dim
        layers.append(nn.Linear(in_dim, num_labels))
        self.classifier = nn.Sequential(*layers)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.classifier(features)


class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance."""

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


print("ClassificationHead and FocalLoss defined.")

In [ ]:
# Cell 14: MultiGuardModel

class MultiGuardModel(nn.Module):
    """Full multimodal model: vision encoder + text encoder + fusion + head."""

    def __init__(
        self,
        vision_encoder: nn.Module,
        text_encoder: nn.Module,
        fusion: nn.Module,
        head: nn.Module,
        loss_fn: nn.Module | None = None,
    ):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.text_encoder = text_encoder
        self.fusion = fusion
        self.head = head
        self.loss_fn = loss_fn or nn.CrossEntropyLoss()

    def forward(self, batch: dict) -> dict:
        """Forward pass through the full pipeline.

        Returns dict with 'logits', 'loss', 'fused_features'.
        """
        vision_features = self.vision_encoder(batch["pixel_values"])
        text_features = self.text_encoder(
            batch["input_ids"], batch.get("attention_mask")
        )
        fused = self.fusion(vision_features, text_features)
        logits = self.head(fused)

        loss = None
        if "labels" in batch:
            loss = self.loss_fn(logits, batch["labels"])

        return {
            "logits": logits,
            "loss": loss,
            "fused_features": fused,
        }


print("MultiGuardModel defined.")

In [ ]:
# Cell 15: build_model Factory

def build_model(
    vision_model_id: str,
    text_model_id: str,
    fusion_type: str,
    vision_dim: int,
    text_dim: int,
    num_labels: int = 2,
    hidden_dims: list[int] | None = None,
    fusion_kwargs: dict | None = None,
    use_focal_loss: bool = False,
) -> MultiGuardModel:
    """Build a MultiGuard model from parameters."""
    hidden_dims = hidden_dims or [512, 256]
    fusion_kwargs = fusion_kwargs or {}

    # Vision encoder
    if "clip" in vision_model_id.lower():
        vision_encoder = VisionEncoder(model_id=vision_model_id)
        actual_vision_dim = vision_encoder.output_dim
    else:
        # EfficientNet or other models via AutoModel
        vision_encoder = EfficientNetVisionEncoder(model_id=vision_model_id)
        actual_vision_dim = vision_encoder.output_dim

    # Text encoder
    text_encoder = TextEncoder(model_id=text_model_id, hidden_size=text_dim)

    # Fusion
    if fusion_type == "late_fusion":
        fusion = LateFusion(
            vision_dim=actual_vision_dim, text_dim=text_dim,
            hidden_dims=hidden_dims, **fusion_kwargs,
        )
    elif fusion_type == "cross_attention":
        fusion = CrossAttentionFusion(
            vision_dim=actual_vision_dim, text_dim=text_dim,
            **fusion_kwargs,
        )
    elif fusion_type == "tensor_fusion":
        fusion = TensorFusion(
            vision_dim=actual_vision_dim, text_dim=text_dim,
            **fusion_kwargs,
        )
    else:
        raise ValueError(f"Unknown fusion type: {fusion_type}")

    # Head
    head = ClassificationHead(
        input_dim=fusion.output_dim, num_labels=num_labels,
        hidden_dims=[fusion.output_dim], dropout=0.3,
    )

    # Loss
    loss_fn = FocalLoss() if use_focal_loss else nn.CrossEntropyLoss()

    model = MultiGuardModel(vision_encoder, text_encoder, fusion, head, loss_fn)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(
        f"Model built: {fusion_type}, "
        f"params={total_params/1e6:.1f}M (trainable={trainable_params/1e6:.1f}M)"
    )
    return model


# EfficientNet encoder for distillation student
class EfficientNetVisionEncoder(nn.Module):
    """Vision encoder using EfficientNet (for student model)."""

    def __init__(self, model_id: str = "google/efficientnet-b0"):
        super().__init__()
        self.model_id = model_id
        from transformers import EfficientNetModel
        self.encoder = EfficientNetModel.from_pretrained(model_id)
        self.output_dim = self.encoder.config.hidden_dim  # 1280 for B0
        logger.info(f"EfficientNetVisionEncoder: {model_id}, output_dim={self.output_dim}")

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.encoder(pixel_values=pixel_values)
        # Global average pooling over spatial dims
        pooled = outputs.last_hidden_state.mean(dim=[2, 3])  # [B, hidden_dim]
        return pooled


print("build_model factory and EfficientNetVisionEncoder defined.")

In [ ]:
# Cell 16: Classification Metrics

def compute_classification_metrics(
    predictions: torch.Tensor,
    labels: torch.Tensor,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Compute accuracy, F1, precision, recall, AUROC."""
    if predictions.dim() > 1 and predictions.shape[1] == 2:
        probs = torch.softmax(predictions, dim=-1)[:, 1].cpu().numpy()
        preds = (probs >= threshold).astype(int)
    elif predictions.dim() > 1:
        probs = torch.softmax(predictions, dim=-1).cpu().numpy()
        preds = predictions.argmax(dim=-1).cpu().numpy()
    else:
        probs = torch.sigmoid(predictions).cpu().numpy()
        preds = (probs >= threshold).astype(int)

    labels_np = labels.cpu().numpy()

    metrics = {
        "accuracy": accuracy_score(labels_np, preds),
        "f1": f1_score(labels_np, preds, average="macro", zero_division=0),
        "precision": precision_score(labels_np, preds, average="macro", zero_division=0),
        "recall": recall_score(labels_np, preds, average="macro", zero_division=0),
    }

    try:
        if predictions.dim() > 1 and predictions.shape[1] == 2:
            metrics["auroc"] = roc_auc_score(labels_np, probs)
        else:
            metrics["auroc"] = roc_auc_score(labels_np, probs, multi_class="ovr")
    except ValueError:
        metrics["auroc"] = 0.0

    return metrics


print("compute_classification_metrics defined.")

In [ ]:
# Cell 17: Optimizer & Scheduler

def build_optimizer(
    model: nn.Module,
    name: str = "adamw",
    lr: float = 1e-4,
    weight_decay: float = 0.01,
) -> torch.optim.Optimizer:
    """Build optimizer."""
    params = filter(lambda p: p.requires_grad, model.parameters())
    optimizers = {
        "adam": torch.optim.Adam,
        "adamw": torch.optim.AdamW,
        "sgd": torch.optim.SGD,
    }
    if name not in optimizers:
        raise ValueError(f"Unknown optimizer: {name}")
    return optimizers[name](params, lr=lr, weight_decay=weight_decay)


def build_scheduler(
    optimizer: torch.optim.Optimizer,
    name: str = "cosine",
    num_training_steps: int = 1000,
    warmup_ratio: float = 0.1,
) -> torch.optim.lr_scheduler.LRScheduler:
    """Build learning rate scheduler."""
    warmup_steps = int(num_training_steps * warmup_ratio)

    if name == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_training_steps - warmup_steps
        )
    elif name == "linear":
        scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, total_iters=warmup_steps
        )
    elif name == "step":
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=num_training_steps // 3
        )
    else:
        raise ValueError(f"Unknown scheduler: {name}")

    return scheduler


print("build_optimizer and build_scheduler defined.")

In [ ]:
# Cell 18: MultiGuardTrainer — Enhanced for Kaggle T4 (FP16 + Grad Accumulation)

class MultiGuardTrainer:
    """Trainer with FP16, gradient accumulation, metrics tracking, and checkpointing."""

    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        scheduler: Any | None = None,
        grad_accum_steps: int = 1,
        output_dir: str = "/kaggle/working",
        model_name: str = "model",
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.grad_accum_steps = grad_accum_steps
        self.output_dir = Path(output_dir)
        self.model_name = model_name
        self.device = DEVICE
        self.model.to(self.device)
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.device.type == "cuda")
        self.best_metric = 0.0
        self.history = {
            "train_loss": [], "val_loss": [],
            "val_accuracy": [], "val_f1": [], "val_auroc": [],
        }

    def train_epoch(self, epoch: int) -> float:
        """Run one training epoch with FP16 and gradient accumulation."""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(self.train_loader):
            batch = {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=self.device.type == "cuda"):
                outputs = self.model(batch)
                loss = outputs["loss"] / self.grad_accum_steps

            self.scaler.scale(loss).backward()

            if (step + 1) % self.grad_accum_steps == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
                if self.scheduler:
                    self.scheduler.step()

            total_loss += loss.item() * self.grad_accum_steps
            num_batches += 1

        avg_loss = total_loss / max(num_batches, 1)
        return avg_loss

    @torch.no_grad()
    def validate(self) -> dict[str, float]:
        """Run validation and compute metrics."""
        self.model.eval()
        total_loss = 0.0
        all_logits = []
        all_labels = []

        for batch in self.val_loader:
            batch = {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=self.device.type == "cuda"):
                outputs = self.model(batch)

            if outputs["loss"] is not None:
                total_loss += outputs["loss"].item()
            all_logits.append(outputs["logits"].cpu())
            all_labels.append(batch["labels"].cpu())

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        metrics = compute_classification_metrics(all_logits, all_labels)
        metrics["val_loss"] = total_loss / max(len(self.val_loader), 1)
        return metrics

    def save_checkpoint(self, metrics: dict[str, float]) -> None:
        """Save best model checkpoint."""
        path = self.output_dir / f"{self.model_name}_best.pt"
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "metrics": metrics,
        }, path)
        logger.info(f"Checkpoint saved: {path}")

    def train(self, num_epochs: int) -> dict:
        """Full training loop."""
        logger.info(f"Starting training: {self.model_name}, {num_epochs} epochs")

        for epoch in range(1, num_epochs + 1):
            t0 = time.time()
            train_loss = self.train_epoch(epoch)
            val_metrics = self.validate()
            elapsed = time.time() - t0

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_metrics["val_loss"])
            self.history["val_accuracy"].append(val_metrics["accuracy"])
            self.history["val_f1"].append(val_metrics["f1"])
            self.history["val_auroc"].append(val_metrics.get("auroc", 0.0))

            logger.info(
                f"Epoch {epoch}/{num_epochs} ({elapsed:.0f}s) | "
                f"train_loss={train_loss:.4f} | val_loss={val_metrics['val_loss']:.4f} | "
                f"acc={val_metrics['accuracy']:.4f} | f1={val_metrics['f1']:.4f} | "
                f"auroc={val_metrics.get('auroc', 0):.4f}"
            )

            # Save best by AUROC
            if val_metrics.get("auroc", 0) > self.best_metric:
                self.best_metric = val_metrics["auroc"]
                self.save_checkpoint(val_metrics)

            log_vram_usage(f"epoch_{epoch}")

        logger.info(f"Training complete. Best AUROC: {self.best_metric:.4f}")
        return self.history


print("MultiGuardTrainer defined.")

---
## Stage 1: Baseline Training

**Architecture:** CLIP ViT-B/16 (vision) + RoBERTa-base (text) + Late Fusion + Classification Head

This is the simplest multimodal approach:
1. Extract image features using CLIP's vision encoder
2. Extract text features using RoBERTa
3. Concatenate features and pass through MLP (Late Fusion)
4. Classify as hateful (1) or not hateful (0)

**Hyperparameters:** batch=16, grad_accum=4 (effective=64), lr=1e-4, 10 epochs

In [ ]:
# Cell 20: Stage 1 — Build & Train Baseline

cfg = CONFIG["baseline"]

# Build model
baseline_model = build_model(
    vision_model_id=cfg["vision_model"],
    text_model_id=cfg["text_model"],
    fusion_type=cfg["fusion"],
    vision_dim=cfg["vision_dim"],
    text_dim=cfg["text_dim"],
    hidden_dims=cfg["hidden_dims"],
    num_labels=cfg["num_labels"],
    use_focal_loss=True,
)

# DataLoaders
train_loader, val_loader, test_loader = create_loaders(batch_size=cfg["batch_size"])

# Optimizer & Scheduler
num_steps = len(train_loader) * cfg["epochs"] // cfg["grad_accum_steps"]
optimizer = build_optimizer(baseline_model, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
scheduler = build_scheduler(optimizer, name="cosine", num_training_steps=num_steps, warmup_ratio=cfg["warmup_ratio"])

# Train
baseline_trainer = MultiGuardTrainer(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    grad_accum_steps=cfg["grad_accum_steps"],
    model_name="baseline_late_fusion",
)

baseline_history = baseline_trainer.train(num_epochs=cfg["epochs"])

In [ ]:
# Cell 21: Plot Baseline Training Curves

def plot_training_curves(history: dict, title: str = "Training Curves"):
    """Plot loss, accuracy, F1, and AUROC over epochs."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss
    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss")
    axes[0].plot(epochs, history["val_loss"], "r-o", label="Val Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{title} — Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy & F1
    axes[1].plot(epochs, history["val_accuracy"], "g-o", label="Accuracy")
    axes[1].plot(epochs, history["val_f1"], "m-o", label="F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title(f"{title} — Accuracy & F1")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # AUROC
    axes[2].plot(epochs, history["val_auroc"], "c-o", label="AUROC")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("AUROC")
    axes[2].set_title(f"{title} — AUROC")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(CONFIG["output_dir"]) / f"{title.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()


plot_training_curves(baseline_history, "Baseline (Late Fusion)")

---
## Stage 2: Fusion Ablation Study

Compare three fusion strategies with the same backbone:
1. **Late Fusion** (baseline) — Simple concatenation + MLP
2. **Cross-Attention Fusion** — Bidirectional attention: vision attends to text and vice versa, with learned gating
3. **Tensor Fusion** — Low-rank factorized outer product to capture cross-modal interactions

**Hyperparameters:** batch=8, grad_accum=8 (effective=64), lr=5e-5, 15 epochs

In [ ]:
# Cell 23: Train Cross-Attention Fusion Model

# Free baseline model memory
del baseline_model, baseline_trainer, optimizer, scheduler
torch.cuda.empty_cache()
log_vram_usage("after_baseline_cleanup")

fa_cfg = CONFIG["fusion_ablation"]
ca_params = fa_cfg["cross_attention"]

cross_attn_model = build_model(
    vision_model_id=CONFIG["baseline"]["vision_model"],
    text_model_id=CONFIG["baseline"]["text_model"],
    fusion_type="cross_attention",
    vision_dim=CONFIG["baseline"]["vision_dim"],
    text_dim=CONFIG["baseline"]["text_dim"],
    num_labels=CONFIG["baseline"]["num_labels"],
    fusion_kwargs=ca_params,
    use_focal_loss=True,
)

train_loader, val_loader, test_loader = create_loaders(batch_size=fa_cfg["batch_size"])
num_steps = len(train_loader) * fa_cfg["epochs"] // fa_cfg["grad_accum_steps"]
optimizer = build_optimizer(cross_attn_model, lr=fa_cfg["lr"])
scheduler = build_scheduler(optimizer, name="cosine", num_training_steps=num_steps)

ca_trainer = MultiGuardTrainer(
    model=cross_attn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    grad_accum_steps=fa_cfg["grad_accum_steps"],
    model_name="cross_attention_fusion",
)

ca_history = ca_trainer.train(num_epochs=fa_cfg["epochs"])

In [ ]:
# Cell 24: Train Tensor Fusion Model

del cross_attn_model, ca_trainer, optimizer, scheduler
torch.cuda.empty_cache()
log_vram_usage("after_cross_attn_cleanup")

tf_params = fa_cfg["tensor_fusion"]

tensor_fusion_model = build_model(
    vision_model_id=CONFIG["baseline"]["vision_model"],
    text_model_id=CONFIG["baseline"]["text_model"],
    fusion_type="tensor_fusion",
    vision_dim=CONFIG["baseline"]["vision_dim"],
    text_dim=CONFIG["baseline"]["text_dim"],
    num_labels=CONFIG["baseline"]["num_labels"],
    fusion_kwargs=tf_params,
    use_focal_loss=True,
)

train_loader, val_loader, test_loader = create_loaders(batch_size=fa_cfg["batch_size"])
num_steps = len(train_loader) * fa_cfg["epochs"] // fa_cfg["grad_accum_steps"]
optimizer = build_optimizer(tensor_fusion_model, lr=fa_cfg["lr"])
scheduler = build_scheduler(optimizer, name="cosine", num_training_steps=num_steps)

tf_trainer = MultiGuardTrainer(
    model=tensor_fusion_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    grad_accum_steps=fa_cfg["grad_accum_steps"],
    model_name="tensor_fusion",
)

tf_history = tf_trainer.train(num_epochs=fa_cfg["epochs"])

In [ ]:
# Cell 25: Compare All 3 Fusion Strategies

def compare_fusions(histories: dict[str, dict]):
    """Grouped bar chart comparing fusion strategies."""
    metrics = ["val_accuracy", "val_f1", "val_auroc"]
    labels = ["Accuracy", "F1", "AUROC"]
    model_names = list(histories.keys())
    colors = ["#2196F3", "#FF9800", "#4CAF50"]

    # Get best epoch metrics for each model
    best_metrics = {}
    for name, hist in histories.items():
        best_epoch = np.argmax(hist["val_auroc"])
        best_metrics[name] = {
            m: hist[m][best_epoch] for m in metrics
        }

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(metrics))
    width = 0.25

    for i, name in enumerate(model_names):
        values = [best_metrics[name][m] for m in metrics]
        bars = ax.bar(x + i * width, values, width, label=name, color=colors[i], alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)

    ax.set_xlabel("Metric")
    ax.set_ylabel("Score")
    ax.set_title("Fusion Strategy Comparison (Best Epoch)")
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(Path(CONFIG["output_dir"]) / "fusion_comparison.png", dpi=150)
    plt.show()

    # Print summary table
    print("\nFusion Comparison Summary:")
    print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'AUROC':>10}")
    print("-" * 55)
    for name, m in best_metrics.items():
        print(f"{name:<25} {m['val_accuracy']:>10.4f} {m['val_f1']:>10.4f} {m['val_auroc']:>10.4f}")


compare_fusions({
    "Late Fusion": baseline_history,
    "Cross-Attention": ca_history,
    "Tensor Fusion": tf_history,
})

---
## Stage 3: Knowledge Distillation

**Goal:** Compress the best teacher model into a smaller, faster student model.

**Teacher:** Best model from fusion ablation (loaded from checkpoint)
**Student:** EfficientNet-B0 (vision) + DistilRoBERTa (text) + Late Fusion

**Distillation Loss:**
- `L = alpha * CE(student, labels) + (1-alpha) * T^2 * KL(student/T, teacher/T)`
- Temperature T=4.0 softens probability distributions
- alpha=0.5 balances hard labels vs soft teacher knowledge

**Hyperparameters:** batch=32, grad_accum=2 (effective=64), lr=1e-3, 20 epochs

In [ ]:
# Cell 27: DistillationTrainer

class DistillationTrainer(MultiGuardTrainer):
    """Knowledge distillation trainer: teacher guides student learning."""

    def __init__(
        self,
        teacher_model: nn.Module,
        temperature: float = 4.0,
        alpha: float = 0.5,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.teacher_model = teacher_model.to(self.device)
        self.teacher_model.eval()
        for param in self.teacher_model.parameters():
            param.requires_grad = False
        self.temperature = temperature
        self.alpha = alpha
        logger.info(f"DistillationTrainer: temp={temperature}, alpha={alpha}")

    def compute_distillation_loss(
        self,
        student_logits: torch.Tensor,
        teacher_logits: torch.Tensor,
        labels: torch.Tensor,
    ) -> torch.Tensor:
        """Combined hard + soft distillation loss."""
        hard_loss = F.cross_entropy(student_logits, labels)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=-1)
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=-1)
        soft_loss = F.kl_div(soft_student, soft_teacher, reduction="batchmean")
        soft_loss = soft_loss * (self.temperature ** 2)
        return self.alpha * hard_loss + (1 - self.alpha) * soft_loss

    def train_epoch(self, epoch: int) -> float:
        """Training epoch with distillation loss."""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(self.train_loader):
            batch = {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=self.device.type == "cuda"):
                # Student forward
                student_outputs = self.model(batch)
                # Teacher forward (no grad)
                with torch.no_grad():
                    teacher_outputs = self.teacher_model(batch)

                loss = self.compute_distillation_loss(
                    student_outputs["logits"],
                    teacher_outputs["logits"],
                    batch["labels"],
                ) / self.grad_accum_steps

            self.scaler.scale(loss).backward()

            if (step + 1) % self.grad_accum_steps == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
                if self.scheduler:
                    self.scheduler.step()

            total_loss += loss.item() * self.grad_accum_steps
            num_batches += 1

        return total_loss / max(num_batches, 1)


print("DistillationTrainer defined.")

In [ ]:
# Cell 28: Load Best Teacher & Train Student

# Clean up previous models
del tensor_fusion_model, tf_trainer, optimizer, scheduler
torch.cuda.empty_cache()

dist_cfg = CONFIG["distillation"]

# Determine best teacher from fusion comparison
# Compare best AUROC from each fusion
fusion_best = {
    "baseline_late_fusion": max(baseline_history["val_auroc"]) if baseline_history["val_auroc"] else 0,
    "cross_attention_fusion": max(ca_history["val_auroc"]) if ca_history["val_auroc"] else 0,
    "tensor_fusion": max(tf_history["val_auroc"]) if tf_history["val_auroc"] else 0,
}
best_teacher_name = max(fusion_best, key=fusion_best.get)
print(f"Best teacher: {best_teacher_name} (AUROC={fusion_best[best_teacher_name]:.4f})")

# Determine fusion type for rebuilding teacher
teacher_fusion_map = {
    "baseline_late_fusion": ("late_fusion", {}),
    "cross_attention_fusion": ("cross_attention", CONFIG["fusion_ablation"]["cross_attention"]),
    "tensor_fusion": ("tensor_fusion", CONFIG["fusion_ablation"]["tensor_fusion"]),
}
teacher_fusion_type, teacher_fusion_kwargs = teacher_fusion_map[best_teacher_name]

# Rebuild and load teacher
teacher_model = build_model(
    vision_model_id=CONFIG["baseline"]["vision_model"],
    text_model_id=CONFIG["baseline"]["text_model"],
    fusion_type=teacher_fusion_type,
    vision_dim=CONFIG["baseline"]["vision_dim"],
    text_dim=CONFIG["baseline"]["text_dim"],
    num_labels=CONFIG["baseline"]["num_labels"],
    fusion_kwargs=teacher_fusion_kwargs,
)
ckpt_path = Path(CONFIG["output_dir"]) / f"{best_teacher_name}_best.pt"
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=True)
teacher_model.load_state_dict(checkpoint["model_state_dict"])
print(f"Teacher loaded from {ckpt_path}")

# Build student model (smaller)
student_model = build_model(
    vision_model_id=dist_cfg["student_vision_model"],
    text_model_id=dist_cfg["student_text_model"],
    fusion_type="late_fusion",
    vision_dim=dist_cfg["student_vision_dim"],
    text_dim=dist_cfg["student_text_dim"],
    num_labels=CONFIG["baseline"]["num_labels"],
    hidden_dims=[256, 128],
)

teacher_params = sum(p.numel() for p in teacher_model.parameters()) / 1e6
student_params = sum(p.numel() for p in student_model.parameters()) / 1e6
print(f"Teacher: {teacher_params:.1f}M params")
print(f"Student: {student_params:.1f}M params")
print(f"Compression ratio: {teacher_params / student_params:.1f}x")

# DataLoaders with distillation batch size
train_loader, val_loader, test_loader = create_loaders(batch_size=dist_cfg["batch_size"])
num_steps = len(train_loader) * dist_cfg["epochs"] // dist_cfg["grad_accum_steps"]
optimizer = build_optimizer(student_model, lr=dist_cfg["lr"])
scheduler = build_scheduler(optimizer, name="cosine", num_training_steps=num_steps)

# Train with distillation
distill_trainer = DistillationTrainer(
    teacher_model=teacher_model,
    temperature=dist_cfg["temperature"],
    alpha=dist_cfg["alpha"],
    model=student_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    grad_accum_steps=dist_cfg["grad_accum_steps"],
    model_name="student_distilled",
)

distill_history = distill_trainer.train(num_epochs=dist_cfg["epochs"])

In [ ]:
# Cell 29: Teacher vs Student Comparison

def compare_teacher_student(teacher_model, student_model, test_loader, device):
    """Compare teacher and student on metrics, model size, and inference speed."""

    def evaluate_model(model, loader, device):
        model.eval()
        model.to(device)
        all_logits, all_labels = [], []
        times = []

        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                t0 = time.time()
                with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                    outputs = model(batch)
                torch.cuda.synchronize() if device.type == "cuda" else None
                times.append(time.time() - t0)
                all_logits.append(outputs["logits"].cpu())
                all_labels.append(batch["labels"].cpu())

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        metrics = compute_classification_metrics(all_logits, all_labels)
        metrics["avg_batch_time_ms"] = np.mean(times) * 1000
        metrics["params_M"] = sum(p.numel() for p in model.parameters()) / 1e6

        # Model size on disk
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".pt") as f:
            torch.save(model.state_dict(), f.name)
            metrics["size_MB"] = os.path.getsize(f.name) / (1024**2)

        return metrics

    teacher_metrics = evaluate_model(teacher_model, test_loader, device)
    student_metrics = evaluate_model(student_model, test_loader, device)

    print("\nTeacher vs Student Comparison:")
    print(f"{'Metric':<25} {'Teacher':>12} {'Student':>12} {'Ratio':>10}")
    print("-" * 60)

    for key in ["accuracy", "f1", "auroc", "params_M", "size_MB", "avg_batch_time_ms"]:
        t_val = teacher_metrics[key]
        s_val = student_metrics[key]
        ratio = s_val / t_val if t_val > 0 else 0
        print(f"{key:<25} {t_val:>12.4f} {s_val:>12.4f} {ratio:>10.2f}x")

    return teacher_metrics, student_metrics


_, test_loader_eval = create_loaders(batch_size=32)[1:]
teacher_metrics, student_metrics = compare_teacher_student(
    teacher_model, student_model, test_loader, DEVICE
)

In [ ]:
# Cell 30: Full Test Set Evaluation for All Models

@torch.no_grad()
def full_evaluation(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple:
    """Run full evaluation, returning logits, labels, and metrics."""
    model.eval()
    model.to(device)
    all_logits, all_labels = [], []

    for batch in loader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            outputs = model(batch)
        all_logits.append(outputs["logits"].cpu())
        all_labels.append(batch["labels"].cpu())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    metrics = compute_classification_metrics(all_logits, all_labels)
    return all_logits, all_labels, metrics


# Load all best checkpoints and evaluate
test_results = {}
checkpoint_names = ["baseline_late_fusion", "cross_attention_fusion", "tensor_fusion", "student_distilled"]

for ckpt_name in checkpoint_names:
    ckpt_path = Path(CONFIG["output_dir"]) / f"{ckpt_name}_best.pt"
    if not ckpt_path.exists():
        print(f"Skipping {ckpt_name} — no checkpoint found.")
        continue

    # Determine fusion type
    if "cross_attention" in ckpt_name:
        ft, fk = "cross_attention", CONFIG["fusion_ablation"]["cross_attention"]
        vid, tid = CONFIG["baseline"]["vision_model"], CONFIG["baseline"]["text_model"]
        vd, td = CONFIG["baseline"]["vision_dim"], CONFIG["baseline"]["text_dim"]
        hd = None
    elif "tensor" in ckpt_name:
        ft, fk = "tensor_fusion", CONFIG["fusion_ablation"]["tensor_fusion"]
        vid, tid = CONFIG["baseline"]["vision_model"], CONFIG["baseline"]["text_model"]
        vd, td = CONFIG["baseline"]["vision_dim"], CONFIG["baseline"]["text_dim"]
        hd = None
    elif "student" in ckpt_name:
        ft, fk = "late_fusion", {}
        vid = CONFIG["distillation"]["student_vision_model"]
        tid = CONFIG["distillation"]["student_text_model"]
        vd = CONFIG["distillation"]["student_vision_dim"]
        td = CONFIG["distillation"]["student_text_dim"]
        hd = [256, 128]
    else:
        ft, fk = "late_fusion", {}
        vid, tid = CONFIG["baseline"]["vision_model"], CONFIG["baseline"]["text_model"]
        vd, td = CONFIG["baseline"]["vision_dim"], CONFIG["baseline"]["text_dim"]
        hd = CONFIG["baseline"]["hidden_dims"]

    eval_model = build_model(
        vision_model_id=vid, text_model_id=tid,
        fusion_type=ft, vision_dim=vd, text_dim=td,
        hidden_dims=hd, fusion_kwargs=fk,
    )
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    eval_model.load_state_dict(ckpt["model_state_dict"])

    logits, labels, metrics = full_evaluation(eval_model, test_loader, DEVICE)
    test_results[ckpt_name] = {"logits": logits, "labels": labels, "metrics": metrics}
    print(f"{ckpt_name}: acc={metrics['accuracy']:.4f}, f1={metrics['f1']:.4f}, auroc={metrics.get('auroc',0):.4f}")

    del eval_model
    torch.cuda.empty_cache()

In [ ]:
# Cell 31: Confusion Matrix, ROC Curve, Classification Report

def plot_evaluation(test_results: dict):
    """Plot confusion matrices and ROC curves for all models."""
    n_models = len(test_results)
    if n_models == 0:
        print("No results to plot.")
        return

    # ROC Curves
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    colors = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63"]

    # Plot ROC curves
    for i, (name, res) in enumerate(test_results.items()):
        logits = res["logits"]
        labels = res["labels"].numpy()
        probs = torch.softmax(logits, dim=-1)[:, 1].numpy()
        fpr, tpr, _ = roc_curve(labels, probs)
        auroc = res["metrics"].get("auroc", 0)
        axes[0].plot(fpr, tpr, color=colors[i % len(colors)],
                     label=f"{name} (AUROC={auroc:.3f})", linewidth=2)

    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curves")
    axes[0].legend(loc="lower right", fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Confusion matrix for best model
    best_name = max(test_results, key=lambda k: test_results[k]["metrics"].get("auroc", 0))
    best_res = test_results[best_name]
    probs = torch.softmax(best_res["logits"], dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(best_res["labels"].numpy(), preds)

    im = axes[1].imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    axes[1].set_title(f"Confusion Matrix — {best_name}")
    plt.colorbar(im, ax=axes[1])
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(["Not Hateful", "Hateful"])
    axes[1].set_yticklabels(["Not Hateful", "Hateful"])

    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=16)

    plt.tight_layout()
    plt.savefig(Path(CONFIG["output_dir"]) / "evaluation_plots.png", dpi=150)
    plt.show()

    # Classification report for best model
    print(f"\nClassification Report — {best_name}:")
    print(classification_report(
        best_res["labels"].numpy(), preds,
        target_names=["Not Hateful", "Hateful"]
    ))


plot_evaluation(test_results)

In [ ]:
# Cell 32: Summary Table

def print_summary_table(test_results: dict):
    """Print comprehensive summary of all models."""
    print("\n" + "=" * 90)
    print("MultiGuard Training Results Summary")
    print("=" * 90)
    print(f"{'Model':<28} {'Params(M)':>10} {'AUROC':>8} {'F1':>8} {'Acc':>8} {'Size(MB)':>10}")
    print("-" * 90)

    for name, res in test_results.items():
        m = res["metrics"]
        # Estimate params from checkpoint
        ckpt_path = Path(CONFIG["output_dir"]) / f"{name}_best.pt"
        size_mb = ckpt_path.stat().st_size / (1024**2) if ckpt_path.exists() else 0

        # Quick param count from checkpoint
        if ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
            params = sum(v.numel() for v in ckpt["model_state_dict"].values()) / 1e6
        else:
            params = 0

        print(f"{name:<28} {params:>10.1f} {m.get('auroc',0):>8.4f} {m['f1']:>8.4f} {m['accuracy']:>8.4f} {size_mb:>10.1f}")

    print("=" * 90)


print_summary_table(test_results)

In [ ]:
# Cell 33: Save Best Model

# Find overall best model
best_overall = max(test_results, key=lambda k: test_results[k]["metrics"].get("auroc", 0))
best_ckpt = Path(CONFIG["output_dir"]) / f"{best_overall}_best.pt"
final_path = Path(CONFIG["output_dir"]) / "multiguard_best.pt"

# Copy best checkpoint as final model
import shutil
shutil.copy2(best_ckpt, final_path)

best_m = test_results[best_overall]["metrics"]
print(f"Best model: {best_overall}")
print(f"  AUROC: {best_m.get('auroc', 0):.4f}")
print(f"  F1:    {best_m['f1']:.4f}")
print(f"  Acc:   {best_m['accuracy']:.4f}")
print(f"Saved to: {final_path}")
print(f"Size: {final_path.stat().st_size / (1024**2):.1f} MB")

In [ ]:
# Cell 34: ONNX Export with Quantization (Optional)

try:
    import onnx
    from torch.quantization import quantize_dynamic

    # Load best model for export
    # Use student if available (smaller), else best teacher
    export_name = "student_distilled" if "student_distilled" in test_results else best_overall

    if "student" in export_name:
        export_model = build_model(
            vision_model_id=CONFIG["distillation"]["student_vision_model"],
            text_model_id=CONFIG["distillation"]["student_text_model"],
            fusion_type="late_fusion",
            vision_dim=CONFIG["distillation"]["student_vision_dim"],
            text_dim=CONFIG["distillation"]["student_text_dim"],
            hidden_dims=[256, 128],
        )
    else:
        ft = "late_fusion"
        fk = {}
        if "cross" in export_name:
            ft, fk = "cross_attention", CONFIG["fusion_ablation"]["cross_attention"]
        elif "tensor" in export_name:
            ft, fk = "tensor_fusion", CONFIG["fusion_ablation"]["tensor_fusion"]
        export_model = build_model(
            vision_model_id=CONFIG["baseline"]["vision_model"],
            text_model_id=CONFIG["baseline"]["text_model"],
            fusion_type=ft, vision_dim=CONFIG["baseline"]["vision_dim"],
            text_dim=CONFIG["baseline"]["text_dim"],
            fusion_kwargs=fk,
        )

    ckpt = torch.load(
        Path(CONFIG["output_dir"]) / f"{export_name}_best.pt",
        map_location="cpu", weights_only=True,
    )
    export_model.load_state_dict(ckpt["model_state_dict"])
    export_model.eval()

    # Dynamic quantization (int8 for linear layers)
    quantized_model = quantize_dynamic(
        export_model, {nn.Linear}, dtype=torch.qint8
    )

    # Save quantized PyTorch model
    quant_path = Path(CONFIG["output_dir"]) / "multiguard_quantized.pt"
    torch.save(quantized_model.state_dict(), quant_path)
    print(f"Quantized model saved: {quant_path}")
    print(f"Quantized size: {quant_path.stat().st_size / (1024**2):.1f} MB")

    del export_model, quantized_model

except ImportError:
    print("ONNX not available. Skipping export. Install with: pip install onnx")
except Exception as e:
    print(f"Export failed (non-critical): {e}")

print("\nDone! All outputs saved to /kaggle/working/")